# 🚀 Stage 1: PanT-HybridNet Cascade Cropping (Data Preparation)

This notebook mathematically destroys the "99% background imbalance" problem. It computationally scans the massive NIfTI volumes, identifies the Pancreas (using the provided label arrays), calculates a tight 3D Bounding Box around the organ, and saves perfectly localized "Mini-CTs" for Stage 2 (Swin-UNETR) to train on.

In [ ]:
import os, glob, json
import numpy as np
import nibabel as nib
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# CORRECT PATHS — verified from your Drive output
DRIVE_ROOT  = '/content/drive/MyDrive/FYP_Data/PanTS'  # Both CT and segs live here
CROPPED_DIR = '/content/drive/MyDrive/PanTS_Cascade'    # Save Mini-CTs to Drive
os.makedirs(CROPPED_DIR, exist_ok=True)

cts  = glob.glob(f'{DRIVE_ROOT}/PanTS_*/ct.nii.gz')
segs = glob.glob(f'{DRIVE_ROOT}/PanTS_*/segmentations')
tumours = glob.glob(f'{DRIVE_ROOT}/PanTS_*/segmentations/pancreatic_lesion.nii.gz')

print(f'CT scans found:        {len(cts)}')
print(f'Segmentation folders:  {len(segs)}')
print(f'Patients with tumour:  {len(tumours)}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CT scans found:        776
Segmentation folders:  50
Patients with tumour:  50


## 🛠️ The Coarse Bounding Box Algorithm
We dynamically load the structural masks (`pancreas.nii.gz` and `pancreatic_lesion.nii.gz`), merge them, find coordinate extremities (`min`, `max`), apply an anatomical padding margin, and physically slice the large CT matrices.

In [ ]:
def cascade_crop_patient(pid, drive_root, save_dir, margin=15):
    ct_path  = os.path.join(drive_root, pid, 'ct.nii.gz')
    seg_dir  = os.path.join(drive_root, pid, 'segmentations')

    if not os.path.exists(ct_path):
        print(f'  ❌ {pid}: ct.nii.gz not found')
        return False
    if not os.path.exists(seg_dir):
        print(f'  ❌ {pid}: segmentations/ not found')
        return False

    # Load full body CT
    ct_img  = nib.load(ct_path)
    ct_data = ct_img.get_fdata().astype(np.float32)

    # Build label: 1=Pancreas (head+body+tail), 2=Tumour
    label_data = np.zeros(ct_data.shape, dtype=np.uint8)

    pancreas_files = glob.glob(os.path.join(seg_dir, 'pancreas*.nii.gz'))
    has_pancreas = False
    for pf in pancreas_files:
        if 'lesion' not in pf:
            label_data[nib.load(pf).get_fdata() > 0] = 1
            has_pancreas = True

    if not has_pancreas:
        print(f'  ⚠️  {pid}: No pancreas mask. Skipping.')
        return False

    lesion_path = os.path.join(seg_dir, 'pancreatic_lesion.nii.gz')
    has_tumor = False
    if os.path.exists(lesion_path):
        t_mask = nib.load(lesion_path).get_fdata()
        if np.sum(t_mask) > 0:
            label_data[t_mask > 0] = 2
            has_tumor = True

    # Calculate pancreas bounding box
    coords  = np.argwhere(label_data == 1)
    z_min, y_min, x_min = np.min(coords, axis=0)
    z_max, y_max, x_max = np.max(coords, axis=0)

    # Add margin
    z_min = max(0, z_min-margin);  z_max = min(ct_data.shape[0], z_max+margin)
    y_min = max(0, y_min-margin);  y_max = min(ct_data.shape[1], y_max+margin)
    x_min = max(0, x_min-margin);  x_max = min(ct_data.shape[2], x_max+margin)

    # Crop!
    cropped_ct    = ct_data   [z_min:z_max, y_min:y_max, x_min:x_max]
    cropped_label = label_data[z_min:z_max, y_min:y_max, x_min:x_max]

    # Save
    os.makedirs(save_dir, exist_ok=True)
    nib.save(nib.Nifti1Image(cropped_ct,    ct_img.affine), f'{save_dir}/ct_cropped.nii.gz')
    nib.save(nib.Nifti1Image(cropped_label, ct_img.affine), f'{save_dir}/label_cropped.nii.gz')
    json.dump({
        'pid': pid, 'has_tumor': has_tumor,
        'original_shape': list(ct_data.shape),
        'cropped_shape':  list(cropped_ct.shape),
        'bbox': [int(z_min), int(z_max), int(y_min), int(y_max), int(x_min), int(x_max)]
    }, open(f'{save_dir}/bbox.json', 'w'), indent=2)

    print(f'  ✅ {pid} | {ct_data.shape} → {cropped_ct.shape} | Tumour: {has_tumor}')
    return True


## 🚀 Execute Pipeline
Let's crop the dataset iteratively!

In [ ]:
TARGET_PIDS = [
    'PanTS_00000806',   # TUMOUR
    'PanTS_00000871',   # TUMOUR
    'PanTS_00000442',   # NO TUMOUR
    'PanTS_00000717',   # NO TUMOUR
]

print('=' * 60)
print('  STAGE 1: CASCADE CROPPING')
print('=' * 60)
ok = 0
for pid in tqdm(TARGET_PIDS):
    if cascade_crop_patient(pid, DRIVE_ROOT, f'{CROPPED_DIR}/{pid}'):
        ok += 1

print()
print(f'  COMPLETE: {ok}/{len(TARGET_PIDS)} patients cropped!')
print(f'  Output at: {CROPPED_DIR}')
print('=' * 60)


  STAGE 1: CASCADE CROPPING


  0%|          | 0/4 [00:00<?, ?it/s]

  ✅ PanTS_00000806 | (454, 326, 85) → (137, 112, 61) | Tumour: True
  ✅ PanTS_00000871 | (487, 359, 229) → (162, 125, 135) | Tumour: True
  ✅ PanTS_00000442 | (512, 402, 213) → (229, 126, 158) | Tumour: False
  ✅ PanTS_00000717 | (512, 440, 215) → (208, 85, 119) | Tumour: False

  COMPLETE: 4/4 patients cropped!
  Output at: /content/drive/MyDrive/PanTS_Cascade
